In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import skew, kurtosis
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = "../data/processed/clean_obd_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

df.head()

In [ ]:
required_columns = [
    "trip_id",
    "time",
    "speed",
    "rpm",
    "throttle",
    "map",
    "maf",
    "coolant_temp"
]

missing = [c for c in required_columns if c not in df.columns]

print("Missing columns:", missing)

In [ ]:
df = df.sort_values(["trip_id","time"])


# speed m/s
df["speed_mps"] = df["speed"] / 3.6


# acceleration
df["acceleration"] = df.groupby("trip_id")["speed_mps"].diff()


# jerk (derivative of acceleration)
df["jerk"] = df.groupby("trip_id")["acceleration"].diff()


# rpm change
df["rpm_delta"] = df.groupby("trip_id")["rpm"].diff()


# throttle change
df["throttle_delta"] = df.groupby("trip_id")["throttle"].diff()

In [ ]:
#Intent Response Features


df["rpm_per_speed"] = df["rpm"] / (df["speed"] + 1)

df["throttle_per_speed"] = df["throttle"] / (df["speed"] + 1)

df["acc_per_throttle"] = df["acceleration"] / (df["throttle"] + 1)

In [ ]:
#Mechanical Stress Signals


df["cold_engine"] = df["coolant_temp"] < 70

df["high_rpm"] = df["rpm"] > 3500

df["high_map"] = df["map"] > df["map"].quantile(0.90)

df["engine_stress"] = (
    df["cold_engine"] & df["high_rpm"]
).astype(int)

In [ ]:
#Sliding Window Parameters

SAMPLING_RATE = 10   # 10 Hz

WINDOW_SECONDS = 10
WINDOW_SIZE = SAMPLING_RATE * WINDOW_SECONDS   # 100 rows

STRIDE_SECONDS = 5
STRIDE = SAMPLING_RATE * STRIDE_SECONDS        # 50 rows

print("Window Size:", WINDOW_SIZE)
print("Stride:", STRIDE)

In [ ]:
#Feature Extraction Function

from scipy.stats import skew, kurtosis

def extract_features(window):

    features = {}

    columns = [
        "speed",
        "rpm",
        "throttle",
        "map",
        "maf",
        "acceleration",
        "jerk"
    ]

    for col in columns:

        values = window[col].values

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()
        features[f"{col}_max"] = values.max()
        features[f"{col}_min"] = values.min()

        features[f"{col}_skew"] = skew(values)
        features[f"{col}_kurt"] = kurtosis(values)

    return features

In [ ]:
#Sliding Window Feature Extraction

from tqdm import tqdm

feature_rows = []

for trip_id, trip in tqdm(df.groupby("trip_id")):

    trip = trip.sort_values("time").reset_index(drop=True)

    for i in range(0, len(trip) - WINDOW_SIZE, STRIDE):

        window = trip.iloc[i:i+WINDOW_SIZE]

        features = extract_features(window)

        features["trip_id"] = trip_id
        features["start_time"] = window["time"].iloc[0]

        feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)

print("Feature Dataset Shape:", features_df.shape)

features_df.head()

In [ ]:
#Event Features

HARSH_ACCEL = 3      # m/s²
HARSH_BRAKE = -3

df["harsh_acc"] = (df["acceleration"] > HARSH_ACCEL).astype(int)
df["harsh_brake"] = (df["acceleration"] < HARSH_BRAKE).astype(int)

In [ ]:
event_rows = []

for trip_id, trip in tqdm(df.groupby("trip_id")):

    trip = trip.reset_index(drop=True)

    for i in range(0, len(trip) - WINDOW_SIZE, STRIDE):

        window = trip.iloc[i:i+WINDOW_SIZE]

        events = {}

        events["harsh_acc_events"] = window["harsh_acc"].sum()
        events["harsh_brake_events"] = window["harsh_brake"].sum()

        events["trip_id"] = trip_id
        events["start_time"] = window["time"].iloc[0]

        event_rows.append(events)

events_df = pd.DataFrame(event_rows)

In [ ]:
# Mechanical Stress Features

df["cold_engine"] = df["coolant_temp"] < 70
df["high_rpm"] = df["rpm"] > 3500
df["high_map"] = df["map"] > df["map"].quantile(0.9)

df["engine_stress"] = (
    df["cold_engine"] & df["high_rpm"]
).astype(int)

In [ ]:
stress_rows = []

for trip_id, trip in tqdm(df.groupby("trip_id")):

    trip = trip.reset_index(drop=True)

    for i in range(0, len(trip) - WINDOW_SIZE, STRIDE):

        window = trip.iloc[i:i+WINDOW_SIZE]

        stress = {}

        stress["engine_stress_events"] = window["engine_stress"].sum()
        stress["high_map_events"] = window["high_map"].sum()

        stress["trip_id"] = trip_id
        stress["start_time"] = window["time"].iloc[0]

        stress_rows.append(stress)

stress_df = pd.DataFrame(stress_rows)

In [ ]:
# Combine All Features

final_features = features_df.merge(
    events_df,
    on=["trip_id","start_time"]
)

final_features = final_features.merge(
    stress_df,
    on=["trip_id","start_time"]
)

print("Final Feature Dataset Shape:", final_features.shape)

final_features.head(20)

In [ ]:
final_features.describe()

# Some Improvements

In [ ]:
# Fix NaN values

print("NaN before:", final_features.isna().sum().sum())

final_features = final_features.fillna(0)

print("NaN after:", final_features.isna().sum().sum())

In [ ]:
# idle context

final_features["is_idle"] = (
    (final_features["speed_mean"] < 1) &
    (final_features["rpm_mean"] > 500)
).astype(int)

In [ ]:
final_features["engine_off_stop"] = (
    (final_features["speed_mean"] < 1) &
    (final_features["rpm_mean"] < 100)
).astype(int)

In [ ]:
final_features["start_acceleration"] = (
    (final_features["speed_mean"] < 5) &
    (final_features["acceleration_max"] > 2)
).astype(int)

In [ ]:
final_features["aggressive_acc"] = (
    final_features["acceleration_max"] > 3
).astype(int)

In [ ]:
final_features["aggressive_brake"] = (
    final_features["acceleration_min"] < -3
).astype(int)

In [ ]:
final_features["engine_load"] = (
    final_features["map_mean"] /
    (final_features["rpm_mean"] + 1)
)

In [ ]:
final_features["rpm_speed_ratio"] = (
    final_features["rpm_mean"] /
    (final_features["speed_mean"] + 1)
)

In [ ]:
final_features["throttle_smoothness"] = (
    final_features["throttle_std"] /
    (final_features["throttle_mean"] + 1)
)

In [ ]:
final_features["acc_smoothness"] = (
    final_features["acceleration_std"] /
    (abs(final_features["acceleration_mean"]) + 0.01)
)

In [ ]:
final_features["jerk_intensity"] = (
    final_features["jerk_std"]
)

In [ ]:
final_features["cold_start"] = (
    (final_features["engine_stress_events"] > 0)
).astype(int)

In [ ]:
# Behavioral stability
final_features["speed_stability"] = (
    final_features["speed_std"] /
    (final_features["speed_mean"] + 1)
)

# Acceleration bias
final_features["acc_brake_ratio"] = (
    final_features["harsh_acc_events"] /
    (final_features["harsh_brake_events"] + 1)
)

# Engine load
final_features["load_indicator"] = (
    final_features["map_mean"] *
    final_features["rpm_mean"]
)

# Driving aggression
final_features["aggression_score"] = (
    final_features["harsh_acc_events"] +
    final_features["harsh_brake_events"] +
    final_features["aggressive_acc"] +
    final_features["aggressive_brake"]
)

# Driver instability
final_features["control_instability"] = (
    final_features["throttle_smoothness"] +
    final_features["acc_smoothness"] +
    final_features["jerk_intensity"]
)

# RPM-speed correction
final_features["rpm_speed_ratio_norm"] = (
    final_features["rpm_speed_ratio"] /
    final_features["rpm_speed_ratio"].mean()
)

# Throttle-speed relation
final_features["throttle_speed_ratio"] = (
    final_features["throttle_mean"] /
    (final_features["speed_mean"] + 1)
)

# Acceleration variability
final_features["acc_variability"] = (
    final_features["acceleration_std"] /
    (final_features["speed_mean"] + 1)
)

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = final_features.select_dtypes(include="number").columns

feature_cols = feature_cols.drop(["trip_id","start_time"])

scaler = StandardScaler()

final_features[feature_cols] = scaler.fit_transform(
    final_features[feature_cols]
)

In [ ]:
final_features.head(25)

In [ ]:
final_features.shape

In [ ]:
final_features.describe()

In [ ]:
save_path = "../data/features/features_windowed_final.parquet"

final_features.to_parquet(save_path, index=False)

print("Feature dataset saved successfully!")
print("Shape:", final_features.shape)

In [ ]:
final_features.to_csv(
    "../data/features/features_windowed_final.csv",
    index=False
)